# OpenAI Chat Completions Client
Generic client for testing any OpenAI-compatible `/v1/chat/completions` endpoint.

In [1]:
import httpx
import json

BASE_URL = "http://localhost:9000"
ENDPOINT = f"{BASE_URL}/v1/chat/completions"

## 1. List available models

In [2]:
resp = httpx.get(f"{BASE_URL}/v1/models", timeout=10)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

200
{
  "object": "list",
  "data": [
    {
      "id": "google-ai-mode",
      "object": "model",
      "created": 0,
      "owned_by": "googleaimode"
    }
  ]
}


## 2. Single request

In [ ]:
resp = httpx.post(ENDPOINT, json={
    "model": "GPT OSS 120B",
    "messages": [{"role": "user", "content": "Say hello in one word."}]
}, timeout=60)

print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

## 3. Streaming

In [ ]:
with httpx.stream("POST", ENDPOINT, json={
    "model": "GPT OSS 120B",
    "messages": [{"role": "user", "content": "Count from 1 to 5."}],
    "stream": True
}, timeout=60) as resp:
    for line in resp.iter_lines():
        if line:
            print(line)

## 4. Multi-turn conversation

In [ ]:
messages = []

def chat(user_msg):
    messages.append({"role": "user", "content": user_msg})
    resp = httpx.post(ENDPOINT, json={
        "model": "GPT OSS 120B",
        "messages": messages
    }, timeout=60)
    reply = resp.json()["choices"][0]["message"]["content"]
    messages.append({"role": "assistant", "content": reply})
    print(f"User: {user_msg}")
    print(f"Assistant: {reply}\n")

chat("My name is Alice.")
chat("What is my name?")

## 5. Error handling

In [ ]:
resp = httpx.post(ENDPOINT, json={"invalid": True}, timeout=10)
print(resp.status_code)
print(resp.json())

## 6. Parallel tool calls (MCP-style)

In [ ]:
# Real web search via the Parallel MCP server (no fake tools).
#
# Chat Completions *delegates* tool execution to the client: the proxy injects
# the configured MCP tools (mcp.json -> parallel__web_search/web_fetch) and
# returns the model's tool_calls to us; WE execute them against Parallel and
# feed results back. So no client-side `tools` are declared here.
#
# Requires the `mcp` package in this kernel:  pip install mcp
import asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

PARALLEL_MCP = "https://search.parallel.ai/mcp"

async def _call_parallel(tool_name, arguments):
    real = tool_name.split("__", 1)[-1]  # strip the proxy's "parallel__" label
    async with streamablehttp_client(PARALLEL_MCP) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            res = await session.call_tool(real, arguments)
            return "".join(c.text for c in res.content if getattr(c, "type", None) == "text")

def execute_tool(name, args):
    text = asyncio.run(_call_parallel(name, args))
    return text[:4000]  # truncate: the transcript is resent each turn (chat-box limit)

def chat_with_tools(user_message, model="GPT OSS 120B", max_steps=6):
    messages = [{"role": "user", "content": user_message}]
    for step in range(max_steps):
        resp = httpx.post(ENDPOINT, json={"model": model, "messages": messages}, timeout=180)
        data = resp.json()
        if resp.status_code != 200 or "choices" not in data:
            print(f"Error {resp.status_code}: {json.dumps(data)[:300]}")
            return None
        msg = data["choices"][0]["message"]
        finish = data["choices"][0]["finish_reason"]
        print(f"--- Step {step + 1} (finish: {finish}) ---")
        if finish != "tool_calls" or not msg.get("tool_calls"):
            print(f"Final answer:\n{msg.get('content')}")
            return msg.get("content")
        messages.append(msg)
        for tc in msg["tool_calls"]:
            args = json.loads(tc["function"]["arguments"])
            result = execute_tool(tc["function"]["name"], args)
            print(f"  - {tc['function']['name']}({args}) => {result[:120].strip()}…")
            messages.append({"role": "tool", "tool_call_id": tc["id"], "content": result})
    return None

chat_with_tools("Use web search: who won the most recent F1 race, and on what date? Cite the source URL.")

--- Step 1 (finish: stop) ---
Final answer:
The most recent F1 race was the 2026 British Grand Prix at Silverstone, won by Charles Leclerc (Ferrari) on Sunday, 5 July 2026.Result and dateWinner: Charles Leclerc (Ferrari)Race: 2026 British Grand PrixDate: 5 July 2026 (race day)Venue: Silverstone Circuit, United KingdomLeclerc’s victory was notable as Ferrari’s 250th Formula 1 win and ended a 624-day wait for his first victory of the season.SourcesFormula 1 official 2026 race results page:https://www.formula1.com/en/results/2026/racesSky Sports F1 results page (lists Great Britain as the latest result, 3–5 Jul):https://www.skysports.com/f1/resultsMotorsport.com British GP results (race date 2–5 Jul 2026, winner Leclerc):https://www.motorsport.com/f1/results/If you’d like, I can also pull the full podium and fastest lap details from one of those pages.


'The most recent F1 race was the 2026 British Grand Prix at Silverstone, won by Charles Leclerc (Ferrari) on Sunday, 5 July 2026.Result and dateWinner: Charles Leclerc (Ferrari)Race: 2026 British Grand PrixDate: 5 July 2026 (race day)Venue: Silverstone Circuit, United KingdomLeclerc’s victory was notable as Ferrari’s 250th Formula 1 win and ended a 624-day wait for his first victory of the season.SourcesFormula 1 official 2026 race results page:https://www.formula1.com/en/results/2026/racesSky Sports F1 results page (lists Great Britain as the latest result, 3–5 Jul):https://www.skysports.com/f1/resultsMotorsport.com British GP results (race date 2–5 Jul 2026, winner Leclerc):https://www.motorsport.com/f1/results/If you’d like, I can also pull the full podium and fastest lap details from one of those pages.'

## 7. Health check

In [ ]:
try:
    resp = httpx.get(f"{BASE_URL}/docs", timeout=5)
    print(f"Server up: {resp.status_code == 200}")
except Exception as e:
    print(f"Server not reachable: {e}")